In [ ]:
# =============================================================
# AI Stock Market Research Assistant — Bronze Ingestion Pipeline
# File    : pipeline/01_bronze_ingestion.py
# Layer   : Bronze (raw, append-only Delta tables)
# Source  : Massive Stocks API via api.polygon.io
#           (Massive rebranded from Polygon.io Oct 2025 — same key)
# Schedule: Run daily via Databricks Workflow
# =============================================================


## Bronze Layer — Raw Ingestion from Massive/Polygon Stocks API
**Catalog:** `main` | **Schema:** `bronze`

Tables written:
- `main.bronze.raw_companies`
- `main.bronze.raw_price_snapshots`
- `main.bronze.raw_news_articles`

Tickers are loaded dynamically from `main.config.ticker_config`.
To change which tickers are tracked, update that table.


In [ ]:
# 0. Imports and config
import requests
import json
import uuid
import time
from datetime import datetime, date, timedelta
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

BATCH_ID    = str(uuid.uuid4())
RUN_DATE    = date.today().isoformat()
INGESTED_AT = datetime.now().isoformat()

print(f"Batch ID   : {BATCH_ID}")
print(f"Run date   : {RUN_DATE}")
print(f"Ingested at: {INGESTED_AT}")


In [ ]:
# 1. Load API key from Databricks Secrets
API_KEY  = dbutils.secrets.get(scope="capstone", key="massive_api_key")

# NOTE: api.massive.com is blocked on Databricks Free Edition.
# api.polygon.io works — Massive is a rebrand of Polygon.io (Oct 2025).
# Same API key, same endpoints, confirmed working.
BASE_URL = "https://api.polygon.io"

def make_params(extra: dict = None) -> dict:
    params = {"apiKey": API_KEY}
    if extra:
        params.update(extra)
    return params

print(f"API key loaded: {len(API_KEY)} chars")
print(f"Base URL: {BASE_URL}")


In [ ]:
# 2. Load active tickers from Unity Catalog config table
TICKERS = [
    row.ticker for row in
    spark.sql("""
        SELECT ticker
        FROM main.config.ticker_config
        WHERE active = true
        ORDER BY ticker
    """).collect()
]

print(f"Loaded {len(TICKERS)} active tickers from main.config.ticker_config")
print(f"Tickers: {', '.join(TICKERS)}")


In [ ]:
# 3. Helper — safe API call with retry + rate limit handling
# Free tier = 5 requests/min → sleep 13s between ticker calls
def api_get(endpoint: str, params: dict = None, retries: int = 3):
    url = f"{BASE_URL}{endpoint}"
    all_params = make_params(params)
    for attempt in range(1, retries + 1):
        try:
            resp = requests.get(url, params=all_params, timeout=15)
            if resp.status_code == 200:
                return resp.json()
            elif resp.status_code == 429:
                wait = 60
                print(f"  Rate limited — waiting {wait}s (attempt {attempt})")
                time.sleep(wait)
            else:
                print(f"  HTTP {resp.status_code} on {endpoint}: {resp.text[:100]}")
                return None
        except requests.RequestException as e:
            print(f"  Request error (attempt {attempt}): {e}")
    return None

RATE_LIMIT_SLEEP = 13  # seconds between ticker calls (free tier: 5 req/min)


In [ ]:
# 4. Setup Bronze schema + idempotent upsert helper
from delta.tables import DeltaTable

spark.sql("CREATE SCHEMA IF NOT EXISTS main.bronze")
print("Schema main.bronze ready")


def upsert_bronze(rows: list, table: str, keys: list, update_existing: bool = True) -> int:
    """Write a batch to Bronze idempotently.

    Bronze stays append-only in spirit — one row per natural key — but re-running
    the same day (a job retry, a manual re-run, a backfill) updates the existing
    row instead of appending a duplicate. Without this, replays inflate Bronze
    row counts and Silver silently hides it behind dedup.

    Keys are the natural grain of each feed:
      raw_companies       -> (ticker, run_date)      one profile per ticker per day
      raw_price_snapshots -> (ticker, snapshot_date) one OHLCV bar per trading day
      raw_news_articles   -> (article_id, ticker)    same article can arrive per ticker

    update_existing=False makes the write insert-if-absent. Use it for immutable
    facts: a published news article never changes, so rewriting the row would
    clobber the batch_id / ingested_at lineage of the run that first saw it.
    Prices and company profiles keep update_existing=True — those keys include
    the date, so a match is a same-day re-fetch and the newer value should win.
    """
    if not rows:
        print(f"No records to write -> {table}")
        return 0

    df = spark.createDataFrame(rows)

    # MERGE errors if two source rows match the same target row, so collapse
    # within-batch duplicates first (e.g. one article returned for two tickers).
    before = df.count()
    df     = df.dropDuplicates(keys)
    n      = df.count()
    if before != n:
        print(f"  Collapsed {before - n} duplicate row(s) within batch on {keys}")

    if not spark.catalog.tableExists(table):
        (df.write.format("delta")
           .mode("append")
           .option("mergeSchema", "true")
           .saveAsTable(table))
        print(f"\nCreated {table} with {n} rows (key: {', '.join(keys)})")
        return n

    # Serverless rejects spark.databricks.delta.schema.autoMerge.enabled, so evolve
    # the target with the supported write option instead — and only when this batch
    # actually introduces a column, to avoid an empty commit on every run.
    new_cols = [c for c in df.columns if c not in set(spark.table(table).columns)]
    if new_cols:
        (df.limit(0).write.format("delta")
           .mode("append")
           .option("mergeSchema", "true")
           .saveAsTable(table))
        print(f"  Schema evolved: added {new_cols}")

    # <=> is null-safe equality, so a null key never silently duplicates.
    cond = " AND ".join(f"t.{k} <=> s.{k}" for k in keys)
    builder = DeltaTable.forName(spark, table).alias("t").merge(df.alias("s"), cond)
    if update_existing:
        builder = builder.whenMatchedUpdateAll()
    builder.whenNotMatchedInsertAll().execute()

    verb = "Upserted" if update_existing else "Inserted new of"
    print(f"\n{verb} {n} rows -> {table} (key: {', '.join(keys)})")
    return n


In [ ]:
# 5. Ingest Company Fundamentals → main.bronze.raw_companies
# Endpoint: GET /v3/reference/tickers/{ticker}
print("\n--- Ingesting company fundamentals ---")
companies_rows = []

for i, ticker in enumerate(TICKERS):
    print(f"  [{i+1}/{len(TICKERS)}] {ticker}", end=" ")
    data = api_get(f"/v3/reference/tickers/{ticker}")

    if data and "results" in data:
        r = data["results"]
        companies_rows.append({
            "batch_id"        : BATCH_ID,
            "run_date"        : RUN_DATE,
            "ticker"          : ticker,
            "name"            : r.get("name"),
            "exchange"        : r.get("primary_exchange"),
            "market_cap"      : float(r["market_cap"]) if r.get("market_cap") else None,
            "description"     : r.get("description"),
            "homepage_url"    : r.get("homepage_url"),
            "total_employees" : r.get("total_employees"),
            "list_date"       : str(r.get("list_date", "")),
            "sic_code"        : str(r.get("sic_code", "")),
            "sic_description" : r.get("sic_description"),
            "locale"          : r.get("locale"),
            "currency_name"   : r.get("currency_name"),
            "active"          : bool(r.get("active", True)),
            "type"            : r.get("type"),
            "raw_json"        : json.dumps(r),
            "ingested_at"     : INGESTED_AT
        })
        print("✓")
    else:
        print("✗ skipped")

    if i < len(TICKERS) - 1:
        time.sleep(RATE_LIMIT_SLEEP)

upsert_bronze(companies_rows, "main.bronze.raw_companies", ["ticker", "run_date"])


In [ ]:
# 6. Ingest OHLCV Price Snapshots → main.bronze.raw_price_snapshots
# Endpoint: GET /v2/aggs/ticker/{ticker}/prev
print("\n--- Ingesting OHLCV price snapshots ---")
price_rows = []

for i, ticker in enumerate(TICKERS):
    print(f"  [{i+1}/{len(TICKERS)}] {ticker}", end=" ")
    data = api_get(f"/v2/aggs/ticker/{ticker}/prev", {"adjusted": "true"})

    if data and "results" in data and len(data["results"]) > 0:
        for r in data["results"]:
            price_rows.append({
                "batch_id"     : BATCH_ID,
                "run_date"     : RUN_DATE,
                "ticker"       : ticker,
                "snapshot_date": RUN_DATE,
                "open"         : float(r["o"])  if r.get("o")  is not None else None,
                "high"         : float(r["h"])  if r.get("h")  is not None else None,
                "low"          : float(r["l"])  if r.get("l")  is not None else None,
                "close"        : float(r["c"])  if r.get("c")  is not None else None,
                "volume"       : float(r["v"])  if r.get("v")  is not None else None,
                "vwap"         : float(r["vw"]) if r.get("vw") is not None else None,
                "transactions" : int(r["n"])    if r.get("n")  is not None else None,
                "timestamp_ms" : int(r["t"])    if r.get("t")  is not None else None,
                "raw_json"     : json.dumps(r),
                "ingested_at"  : INGESTED_AT
            })
        print("✓")
    else:
        print("✗ skipped")

    if i < len(TICKERS) - 1:
        time.sleep(RATE_LIMIT_SLEEP)

upsert_bronze(price_rows, "main.bronze.raw_price_snapshots", ["ticker", "snapshot_date"])


In [ ]:
# 7. Ingest News Articles → main.bronze.raw_news_articles
# Endpoint: GET /v2/reference/news
print("\n--- Ingesting news articles ---")
news_rows = []
week_ago = (date.today() - timedelta(days=7)).isoformat()

for i, ticker in enumerate(TICKERS):
    print(f"  [{i+1}/{len(TICKERS)}] {ticker}", end=" ")
    data = api_get("/v2/reference/news", {
        "ticker"           : ticker,
        "published_utc.gte": week_ago,
        "order"            : "desc",
        "limit"            : 10
    })

    if data and "results" in data:
        articles = data["results"]
        for article in articles:
            insights  = article.get("insights") or []
            sentiment = insights[0].get("sentiment") if insights else None
            news_rows.append({
                "batch_id"      : BATCH_ID,
                "run_date"      : RUN_DATE,
                "ticker"        : ticker,
                "article_id"    : article.get("id"),
                "title"         : article.get("title"),
                "author"        : article.get("author"),
                "published_utc" : article.get("published_utc"),
                "article_url"   : article.get("article_url"),
                "description"   : article.get("description"),
                "keywords"      : json.dumps(article.get("keywords", [])),
                "publisher_name": (article.get("publisher") or {}).get("name"),
                "sentiment"     : sentiment,
                "raw_json"      : json.dumps(article),
                "ingested_at"   : INGESTED_AT
            })
        print(f"✓ ({len(articles)} articles)")
    else:
        print("✗ skipped")

    if i < len(TICKERS) - 1:
        time.sleep(RATE_LIMIT_SLEEP)

# Articles are immutable — insert only, so historical batch_id lineage survives.
upsert_bronze(news_rows, "main.bronze.raw_news_articles", ["article_id", "ticker"],
              update_existing=False)


In [ ]:
# 8. Verification — row counts and data preview
print("\n=== Bronze Ingestion Summary ===")
print(f"Batch ID : {BATCH_ID}")
print(f"Run date : {RUN_DATE}\n")

for table in ["raw_companies", "raw_price_snapshots", "raw_news_articles"]:
    try:
        df       = spark.table(f"main.bronze.{table}")
        total    = df.count()
        this_run = df.filter(f"batch_id = '{BATCH_ID}'").count()
        print(f"  main.bronze.{table:<25} this run: {this_run:>4}  |  total: {total:>6}")
    except Exception:
        print(f"  main.bronze.{table:<25} not yet created")

print("\n--- Sample: Companies ---")
spark.table("main.bronze.raw_companies") \
     .select("ticker", "name", "exchange", "market_cap") \
     .filter(f"batch_id = '{BATCH_ID}'") \
     .show(5, truncate=False)

print("\n--- Sample: Price Snapshots ---")
spark.table("main.bronze.raw_price_snapshots") \
     .select("ticker", "snapshot_date", "open", "high", "low", "close", "volume") \
     .filter(f"batch_id = '{BATCH_ID}'") \
     .show(5)

print("\n--- Sample: News ---")
spark.table("main.bronze.raw_news_articles") \
     .select("ticker", "title", "publisher_name", "sentiment") \
     .filter(f"batch_id = '{BATCH_ID}'") \
     .show(5, truncate=True)

print("\nBronze ingestion complete ✓")
